In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

def get_feature_columns(df):
    cols_to_drop = [
        col for col in df.columns
        if ('lat' in col.lower() or 'lon' in col.lower() or 'date' in col.lower() or 'time' in col.lower())
    ]
    if 'flood' not in df.columns:
        raise ValueError("Target column 'flood' not found in dataset.")
    X = df.drop(columns=cols_to_drop + ['flood'], errors='ignore')
    y = df['flood'].astype(int)
    return X, y, cols_to_drop

def train_flood_model(df):
    X, y, cols_to_drop = get_feature_columns(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        shuffle=True
    )

    rf_model = RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',
        random_state=42,
        max_depth=10,
        min_samples_leaf=2,
        n_jobs=-1
    )

    print("Training the Random Forest model...")
    rf_model.fit(X_train, y_train)

    # Optional leakage sanity check
    y_shuffled = np.random.permutation(y_train)
    rf_model.fit(X_train, y_shuffled)
    shuffle_score = rf_model.score(X_test, y_test)
    print("\nSanity Check (Shuffled Labels Score):", round(shuffle_score, 4))

    # Retrain correctly
    rf_model.fit(X_train, y_train)

    y_train_pred = rf_model.predict(X_train)
    y_test_pred = rf_model.predict(X_test)

    print("\n" + "="*30)
    print("MODEL PERFORMANCE RESULTS")
    print("="*30)
    print(f"Train Accuracy: {accuracy_score(y_train, y_train_pred):.4f}")
    print(f"Test Accuracy : {accuracy_score(y_test, y_test_pred):.4f}")
    print(f"Train F1      : {f1_score(y_train, y_train_pred, zero_division=0):.4f}")
    print(f"Test F1       : {f1_score(y_test, y_test_pred, zero_division=0):.4f}")
    print(f"Test Precision: {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
    print(f"Test Recall   : {recall_score(y_test, y_test_pred, zero_division=0):.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_test_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_test_pred, zero_division=0))

    importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
    print("\nTop 10 Important Features:")
    print(importances.head(10))

    return rf_model, X.columns, importances

def test_on_unseen_with_dates(model, unseen_df, train_columns, threshold=0.5):
    df_copy = unseen_df.copy()

    date_cols = [col for col in unseen_df.columns if any(key in col.lower() for key in ['date', 'time', 'timestamp'])]
    date_col = date_cols[0] if date_cols else None

    cols_to_drop = [
        col for col in unseen_df.columns
        if ('lat' in col.lower() or 'lon' in col.lower() or 'date' in col.lower() or 'time' in col.lower())
    ]

    X_unseen = unseen_df.drop(columns=cols_to_drop + ['flood'], errors='ignore')

    # Align unseen columns to training columns
    X_unseen = X_unseen.reindex(columns=train_columns, fill_value=0)

    if 'flood' in unseen_df.columns:
        y_true = unseen_df['flood'].astype(int)
        has_true_labels = True
    else:
        y_true = None
        has_true_labels = False

    probs = model.predict_proba(X_unseen)[:, 1]
    y_pred = (probs >= threshold).astype(int)

    print("\n" + "="*30)
    print("UNSEEN DATA RESULTS")
    print("="*30)

    print("\nPrediction Distribution:")
    print(pd.Series(y_pred).value_counts().sort_index())

    if has_true_labels:
        print(f"\nAccuracy : {accuracy_score(y_true, y_pred):.4f}")
        print(f"F1 Score : {f1_score(y_true, y_pred, zero_division=0):.4f}")
        print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
        print(f"Recall   : {recall_score(y_true, y_pred, zero_division=0):.4f}")

        print("\nConfusion Matrix:")
        print(confusion_matrix(y_true, y_pred))

        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, zero_division=0))

    print("\n" + "="*30)
    print("🚨 FLOOD ALERTS (Predicted = 1)")
    print("="*30)

    flagged_indices = np.where(y_pred == 1)[0]

    if len(flagged_indices) == 0:
        print("No flood events detected.")
    else:
        for idx in flagged_indices:
            if date_col:
                print(f"Index: {idx} | Date: {df_copy.iloc[idx][date_col]} | Prob: {probs[idx]:.3f}")
            else:
                print(f"Index: {idx} | Prob: {probs[idx]:.3f}")

    return y_pred, probs

# ===============================
# MAIN
# ===============================
try:
    train_path = "/kaggle/input/datasets/arjunmahesh09999/create/newfile (1).csv"
    unseen_path = "/kaggle/input/datasets/arjunmahesh09999/lucknow-unseen/2008 CHANGED.csv"

    train_df = pd.read_csv(train_path)
    unseen_df = pd.read_csv(unseen_path)

    model, train_columns, feature_importances = train_flood_model(train_df)
    y_pred_unseen, probs_unseen = test_on_unseen_with_dates(model, unseen_df, train_columns, threshold=0.5)

except FileNotFoundError as e:
    print("Error: CSV file not found.")
    print(e)
except Exception as e:
    print(f"Error: {e}")

Training the Random Forest model...

Sanity Check (Shuffled Labels Score): 0.9726

MODEL PERFORMANCE RESULTS
Train Accuracy: 0.9989
Test Accuracy : 1.0000
Train F1      : 0.9630
Test F1       : 1.0000
Test Precision: 1.0000
Test Recall   : 1.0000

Confusion Matrix:
[[215   0]
 [  0   4]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       215
           1       1.00      1.00      1.00         4

    accuracy                           1.00       219
   macro avg       1.00      1.00      1.00       219
weighted avg       1.00      1.00      1.00       219


Top 10 Important Features:
cumi_orig_5    0.178777
cumi_adj_3     0.170813
cumi_orig_1    0.138523
cumi_adj_4     0.100281
cumi_orig_4    0.085785
cumi_orig_3    0.071207
cumi_adj_2     0.065702
cumi_adj_1     0.065264
cumi_orig_2    0.041499
cumi_orig_6    0.035567
dtype: float64

UNSEEN DATA RESULTS

Prediction Distribution:
0    344
1     22
Name: count,